In [6]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from libs.market_dashboard import build_dashboard
from libs import data_loader 
from libs import indicators as ind
from libs.cfd_cost import CFDCostModel 
from libs.account import CFDAccountSimulator 
from libs.performance import PerformanceAnalytics
from libs.result_store import ResultStore
from libs.data_manager import run
from libs.data_loader import load_csv
from libs.market_stats import analyze, market_metrics
from strategies.strategy2 import CHOCHFibBacktester
from strategies.strategy_msb import MSBOBFibBacktester
from libs.desk_card import desk_card
from libs.market_sessions import session_name
from libs.intraday_profile import build_report


In [17]:
import vectorbt as vbt

In [19]:
run('polygon') # updating files÷
run('binance') 
run('yfinance')

[INFO] Polygon I:NDX 5m: resume from 2026-07-02 21:15:00
[DONE] Polygon I:NDX 5m: +188 rows -> ./data/marketdata/NDX/5m.csv
[INFO] Polygon I:NDX 15m: resume from 2026-07-02 21:15:00
[DONE] Polygon I:NDX 15m: +64 rows -> ./data/marketdata/NDX/15m.csv
[INFO] Polygon I:NDX 30m: resume from 2026-07-02 21:00:00
[DONE] Polygon I:NDX 30m: +32 rows -> ./data/marketdata/NDX/30m.csv
[INFO] Polygon I:NDX 1h: resume from 2026-07-02 21:00:00
[DONE] Polygon I:NDX 1h: +18 rows -> ./data/marketdata/NDX/1h.csv
[INFO] Polygon I:NDX 2h: resume from 2026-07-02 20:00:00
[DONE] Polygon I:NDX 2h: +11 rows -> ./data/marketdata/NDX/2h.csv
[INFO] Polygon I:NDX 4h: resume from 2026-07-02 18:00:00
[WARN] HTTP 429; backing off 2s
[WARN] HTTP 429; backing off 4s
[WARN] HTTP 429; backing off 8s
[WARN] HTTP 429; backing off 16s
[DONE] Polygon I:NDX 4h: +7 rows -> ./data/marketdata/NDX/4h.csv
[INFO] Polygon C:XAUUSD 5m: resume from 2026-07-05 23:55:00
[DONE] Polygon C:XAUUSD 5m: +555 rows -> ./data/marketdata/XAUUSD/5

In [3]:
asset = "GOLD"
# gold_df = load_csv('./data/marketdata/XAUUSD/1d.csv',True)
# gold_df.set_index('Datetime',inplace=True)
# nas100_df = load_csv('./data/marketdata/NDX/1d.csv',True)
# nas100_df.set_index('Datetime',inplace=True)
df = load_csv('./data/marketdata/NDX/15m.csv',True)
df.set_index('Datetime',inplace=True)
# df = df.loc["2025":"2026"]

In [5]:
build_report(df, "eurusd_profile.html", tz="America/New_York", day_roll=("America/New_York", 17))

'eurusd_profile.html'

In [9]:
session_name(df['Close'].iloc[1])

/Users/woland/workspace/coding/quant_env/libs/market_sessions.py:188: UserWarning:

Discarding nonzero nanoseconds in conversion.



'Sydney|Tokyo'

In [30]:
ten_bond_df.tail()

,Open,High,Low,Close,Volume
Datetime,,,,,
2026-06-26 05:00:00,4.388,4.410,4.367,4.372,0
2026-06-30 05:00:00,0.000,0.000,0.000,4.418,0
2026-07-01 05:00:00,4.483,4.501,4.457,4.475,0
2026-07-02 05:00:00,4.503,4.505,4.453,4.485,0
2026-07-07 05:00:00,4.497,4.533,4.485,4.529,0


In [21]:
ms = analyze(ETH_df, tz="UTC")     # configure once
mm = market_metrics(ETH_df)
# mm
# mm['meta'],
# mm['distribution'],
# mm['volatility'],
# mm['mean_reversion'],
# mm['calendar'],
# mm['sessions'],
# mm['probability'],
# mm['regimes']
# mm['desk']

In [22]:
mm

{'meta': {'name': 'instrument',
  'n_bars': 14281,
  'bars_per_year': 2191.5,
  'start': '2020-01-01 00:00:00',
  'end': '2026-07-08 04:00:00'},
 'distribution': {'n_returns': 14280,
  'mean': 0.000182214553305663,
  'std': 0.01673513171461649,
  'skewness': -0.683445811026792,
  'excess_kurtosis': 12.35093387871844,
  'jarque_bera_stat': 91876.30642783454,
  'jarque_bera_p': 0.0,
  'is_normal_5pct': False,
  'hill_tail_index': 2.996896567267995,
  'student_t_dof': 2.182853648374344,
  'empirical_quantiles': {'0.01': -0.050379796661374925,
   '0.05': -0.025566919725837942,
   '0.25': -0.006230110755908146,
   '0.5': 0.0002707592848730975,
   '0.75': 0.007093268376538724,
   '0.95': 0.025541347955794536,
   '0.99': 0.04655850882338596}},
 'desk': {'params': {'book': 1000000.0,
   'target_vol': 0.1,
   'kelly_fraction': 0.25,
   'hedge_tenor_months': 1.0,
   'bars_per_year': 2191.5,
   'n_returns': 14280},
  'annualized': {'vol': 0.783429412472777,
   'drift': 0.39932319356936047,
   'sh

In [23]:

build_dashboard({
    "GOLD 1D": ETH_df,
},output="./outs/market_dashbashboard.html")  # writes market_dashboard.html and opens it

'/Users/woland/workspace/coding/quant_env/outs/market_dashbashboard.html'

In [ ]:
desk_card(mm, equity=1_000_000, vol_target=0.10, point_value=1.0)


In [ ]:

marketdata_path = "./data/marketdata"
asset = "XAUUSD"
timeframe = "5m"
run_id = f"{asset}_RUN_{timeframe}_1"
df = data_loader.load_csv(f"{marketdata_path}/{asset}/{timeframe}.csv")
df["ATR"] = ind.calculate_atr(df, 14)
trades_df, details = MSBOBFibBacktester(run_id=run_id,
                                        asset_class="C",
                                        entry_fib_level=0.4,
                                        zigzag_len = 4,
                                        timeframe=timeframe).backtest(df)

In [ ]:
trades_df
details['trades'], details['metadata']

In [ ]:
trades_df

In [ ]:
costs = CFDCostModel("XAUUSD", lots=0.1)   
costed = costs.add_costs(trades_df) 
costed, print(costs.summary(costed))

In [ ]:
costed

In [ ]:
sim = CFDAccountSimulator(
        symbol="XAUUSD",
        initial_capital=10_000,
        use_risk_sizing=False, risk_mode="percent", risk_per_trade=0.01,  # 1%/trade
        leverage=20.0,
    )

In [ ]:
result_df, equity_curve = sim.simulate(details["trades"])   # needs sl_price
result_df, equity_curve
print(sim.stats(result_df, equity_curve))

In [ ]:
perf = PerformanceAnalytics(result_df, equity_curve,
                                risk_free_rate=0.04, periods_per_year=252)

In [ ]:
report = perf.report()

In [ ]:
report['equity_curve']

In [ ]:
current_low = 30451.2
pre_low = 30510.4
current_high = 30597.4 
# l1 - (distance between h0 and l1) × 0.33


In [ ]:
pre_low - ( current_high - pre_low) * 0.33

In [ ]:
report['metrics']

In [ ]:
report["exit_reasons"]       # DataFrame ready for a pie chart

In [ ]:
report["monthly_returns"]    # Series ready for a histogram


In [ ]:
report["rolling_sharpe"]     # Series ready to plot

In [ ]:
store = ResultStore()
store.save_pipeline(
        run_id   = run_id,
        asset    = asset,
        backtest    = (trades_df, details),         # CHOCHFibBacktester.backtest(df)
        account     = (result_df, equity_curve),    # CFDAccountSimulator.simulate(...)
        performance = perf.report(),                  # PerformanceAnalytics.report()
        extra_metadata = {"leverage": 20, "initial_capital": 10_000, "risk_per_trade": 0.01},
        overwrite   = True,
    )

store.list_runs(asset=asset)     # → ["XAUUSD_1h_fib50", ...]
run = store.load_run(run_id)   # dict of every saved artifact
store.summary_table()  


In [ ]:
# import sqlite3


In [ ]:
# conn = sqlite3.connect(f"{DB_path}all_backtests.db")

In [ ]:
# dbs = pd.read_sql_query("PRAGMA database_list;", conn)

In [ ]:
# tables = pd.read_sql_query(
#     "SELECT * FROM sqlite_master",
#     conn
# )
# tables

In [ ]:
# user_tables = pd.read_sql_query(
#     "SELECT * FROM artifacts",
#     conn
# )
# user_tables

# in distribution calculate desk walk through 
# 1 . Annulaize 
# 1.1 annualize vol
# 1.2 annualize drift 
# 1.2 annualize sharpe 
# 2. size the position 
# 2.1 book value 1M, target vol = 10% 
# 2.2 wieght 
# 2.3 notional X book size 
# 2.4 daily P&L
# 2.5 annualize P&L 
# 4. set daily loss limit 
# 4.1 confidenve Level - Tail Prob - Quantile - Return - notional  
# 4.2 Expected Freq 
# 4.3 chance of worse Day 
# 5. Expected shortfall 
# 5.1 how bad is bad given it's a bad day Expected short fall calculation 
# 5.1 ES for all confidence level in term of dollar value and percentage of the book 
# 6. Cap leverage using kelly 
# 7 . Crash Insurance
# 7.1 Crash Insurance Cost 

In [ ]:
time-of-day buckets + realized volatility + volume profile + trend/mean-reversion split.





In [ ]:
# 4 main global trading sessions 
# Sydney, Tokyo, London, and New York 
# Sydney session: starts first in the global day cycle; its UTC time depends on Australia’s local DST rules.
# Tokyo session: starts next; Japan does not use DST, so Tokyo’s UTC open is stable across the year
# London session: one of the most important because it overlaps with both Asia tail-end and New York start; its UTC time changes with UK DST
# New York session: the other major driver of volatility; its UTC time changes with U.S. DST.

In [1]:
# 1967
# 1H+ at 2:00 of the last sunday of april , 1H- at 2:00am Last sunday on October 
# beginning October 27, 1974, and ending February 23, 1975, when DST resumed.
# In 1986,
# the first Sunday in April and having the end remain the last Sunday in October
# 2005 
# DST begins on the second Sunday of March and ends on the first Sunday of November.




SyntaxError: invalid decimal literal (706140292.py, line 2)

In [ ]:
# Market times